# 📕 Pre-Workshop Notebook 6: Putting It All Together — Your Complete Research Workflow
## Scientific Workflows for Brain and Behavioral Research
### National Science Foundation Supported Learning Initiative (Award No. OAC-2417875)

---

## 📋 Final Portfolio Setup
*Run the cell below to generate your portfolio registration header.*

In [ ]:
# FINAL PORTFOLIO SETUP
student_name = "Hari Sai Kaja"
institution = "Grand Valley State University"
completion_date = "2026-08-14"

print(f"🎓 Portfolio record generated for: {student_name} ({institution})")
print(f"📅 Date: {completion_date}")
print("✅ Pre-Workshop Notebooks 01 through 05 confirmed complete.")

---

## 🤖 Using AI Tools to Build Your Workflow ("Ask → Build → Document")

As a **Workflow Designer** completing your capstone integration, your final challenge is making sure all four stages of your workflow connect properly. Individual stages designed separately sometimes pass data in formats that the next stage does not expect — causing silent errors that are hard to spot.

When using the **Ask → Build → Document** cycle in this notebook, ask the AI to add clear status messages between each stage so you can confirm that data is being passed correctly from one step to the next.

---
## 🧠 Part A: Building a Complete Four-Stage Research Workflow
### Topic: Connecting All Stages from Data Loading to Visualization

This notebook brings together everything from Notebooks 1 through 4. You will build a single integrated workflow that connects all four stages — Obtaining Data, Analyzing Data, Visualizing Results, and Documenting Research — into one complete, working pipeline.

### 🧭 Activity A.1 — The Complete Integrated Research Workflow

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the code cell beneath it.*
>
> "Act as a lead neuroscience workflow designer helping a research team build their first complete end-to-end analysis pipeline. Write a single Python script that connects four completely independent, clearly separated stages into one working workflow. The script should:
> 1. Stage 1 — Obtaining Data: Generate a simulated 64-channel brain recording with realistic background noise and some sudden movement artifacts. This stage should not perform any analysis or create any plots.
> 2. Stage 2 — Analyzing Data: Accept the raw data from Stage 1 and clean it using an automated threshold to remove the movement artifacts, without modifying the original data. Then extract key summary features from the cleaned signal.
> 3. Stage 3 — Visualizing Results: Accept the processed features from Stage 2 and create a high-quality figure showing the raw versus cleaned signal side-by-side, along with a summary of the extracted features.
> 4. Stage 4 — Documenting Research: Record the analysis settings used, the processing time for each stage, and any errors that were caught — saving this information to a structured log file.
> Add error-handling safeguards around each stage so that if one stage encounters a problem, it logs the issue and the other stages can still run. Include clear status messages between stages confirming that data passed correctly from one stage to the next."

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import json
import time
import os
import traceback # Added import for traceback

def run_neuroscience_workflow():
    workflow_log = {
        "analysis_settings": {
            "n_channels": 64,
            "sampling_rate_hz": 1000,
            "duration_seconds": 10,
            "noise_amplitude": 0.1,
            "artifact_amplitude": 5.0,
            "artifact_probability": 0.05,
            "cleaning_threshold_std": 3.0
        },
        "stage_processing_times": {},
        "errors_caught": []
    }

    print("Starting Neuroscience Workflow...")

    # --- Stage 1: Obtaining Data ---
    stage_name_1 = "Stage 1: Obtaining Data"
    start_time_1 = time.time()
    raw_data = None
    try:
        print(f"\n{'='*50}\n{stage_name_1}: Generating simulated 64-channel brain recording.\n{'='*50}")

        settings = workflow_log["analysis_settings"]
        n_channels = settings["n_channels"]
        sampling_rate_hz = settings["sampling_rate_hz"]
        duration_seconds = settings["duration_seconds"]
        noise_amplitude = settings["noise_amplitude"]
        artifact_amplitude = settings["artifact_amplitude"]
        artifact_probability = settings["artifact_probability"]

        n_samples = sampling_rate_hz * duration_seconds
        time_vector = np.linspace(0, duration_seconds, n_samples, endpoint=False)

        # Simulate background brain activity (e.g., sine waves)
        brain_activity = np.sin(2 * np.pi * 10 * time_vector) * 0.5 + \
                         np.sin(2 * np.pi * 2 * time_vector) * 0.2
        brain_activity = np.tile(brain_activity, (n_channels, 1))

        # Add realistic background noise
        noise = np.random.normal(0, noise_amplitude, size=(n_channels, n_samples))

        raw_data = brain_activity + noise

        # Add sudden movement artifacts to random channels/times
        for i in range(n_channels):
            num_artifacts = np.random.binomial(n_samples // 100, artifact_probability)
            artifact_indices = np.random.choice(n_samples - 10, num_artifacts, replace=False)
            for idx in artifact_indices:
                artifact_shape = artifact_amplitude * np.exp(-np.linspace(-1, 1, 10)**2 * 5)
                raw_data[i, idx:idx+10] += artifact_shape

        print(f"{stage_name_1} completed successfully. Generated data shape: {raw_data.shape}")

    except Exception as e:
        error_msg = f"Error in {stage_name_1}: {e}"
        print(f"FAILURE: {error_msg}")
        workflow_log["errors_caught"].append({"stage": stage_name_1, "message": str(e), "traceback": traceback.format_exc()})

    finally:
        workflow_log["stage_processing_times"][stage_name_1] = time.time() - start_time_1

    # --- Stage 2: Analyzing Data ---
    stage_name_2 = "Stage 2: Analyzing Data"
    start_time_2 = time.time()
    cleaned_data = None
    summary_features = None
    try: # Outer try block to ensure finally for Stage 2 always executes
        if raw_data is not None:
            try:
                print(f"\n{'='*50}\n{stage_name_2}: Cleaning data and extracting features.\n{'='*50}")

                # Ensure not modifying original data
                cleaned_data = raw_data.copy()

                # Automated thresholding to remove movement artifacts
                cleaning_threshold_std = workflow_log["analysis_settings"]["cleaning_threshold_std"]

                # Calculate a robust estimate of noise standard deviation (e.g., median absolute deviation or IQR based)
                # For simplicity, using std dev of the initial portion of data or after a rough median filter
                # A more sophisticated approach would use iterative artifact detection or robust statistics
                for i in range(n_channels):
                    channel_data = cleaned_data[i, :]

                    # Use a rolling median to estimate baseline, then look for deviations
                    # Using a simpler method here for demonstration: global std
                    std_dev = np.std(channel_data)
                    threshold = cleaning_threshold_std * std_dev

                    # Find indices where signal exceeds threshold (absolute value)
                    artifact_indices = np.where(np.abs(channel_data) > threshold)[0]

                    # Simple artifact removal: replace with local mean or interpolation
                    # Here, we'll replace with the mean of surrounding non-artifact data if possible
                    for idx in artifact_indices:
                        if idx > 0 and idx < n_samples - 1:
                            # Replace with the average of neighboring non-artifact points
                            left_val = channel_data[idx-1] if np.abs(channel_data[idx-1]) <= threshold else None
                            right_val = channel_data[idx+1] if np.abs(channel_data[idx+1]) <= threshold else None

                            if left_val is not None and right_val is not None:
                                cleaned_data[i, idx] = (left_val + right_val) / 2
                            elif left_val is not None:
                                cleaned_data[i, idx] = left_val
                            elif right_val is not None:
                                cleaned_data[i, idx] = right_val
                            else:
                                # If both neighbors are artifacts, replace with global mean or 0
                                cleaned_data[i, idx] = 0 # Fallback
                        else:
                            cleaned_data[i, idx] = 0 # Edge cases

                # Extract key summary features from the cleaned signal
                summary_features = {
                    "mean_amplitude": np.mean(cleaned_data, axis=1).tolist(),
                    "std_amplitude": np.std(cleaned_data, axis=1).tolist(),
                    "peak_to_peak": (np.max(cleaned_data, axis=1) - np.min(cleaned_data, axis=1)).tolist()
                }
                print(f"{stage_name_2} completed successfully. Extracted features for {n_channels} channels.")
                print("Data passed from Stage 1 to Stage 2 successfully.")

            except Exception as e:
                error_msg = f"Error in {stage_name_2}: {e}"
                print(f"FAILURE: {error_msg}")
                workflow_log["errors_caught"].append({"stage": stage_name_2, "message": str(e), "traceback": traceback.format_exc()})
        else:
            print(f"Skipping {stage_name_2} due to previous errors.")
    finally: # This finally belongs to the outer try and handles timing for Stage 2
        workflow_log["stage_processing_times"][stage_name_2] = time.time() - start_time_2

    # --- Stage 3: Visualizing Results ---
    stage_name_3 = "Stage 3: Visualizing Results"
    start_time_3 = time.time()
    try: # Outer try block to ensure finally for Stage 3 always executes
        if raw_data is not None and cleaned_data is not None and summary_features is not None:
            try:
                print(f"\n{'='*50}\n{stage_name_3}: Creating high-quality figure.\n{'='*50}")

                fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 10), sharex=True)

                # Plot raw vs. cleaned signal for a representative channel (e.g., channel 0)
                channel_to_plot = 0
                axes[0].plot(time_vector, raw_data[channel_to_plot, :], label='Raw Signal', alpha=0.7)
                axes[0].plot(time_vector, cleaned_data[channel_to_plot, :], label='Cleaned Signal', alpha=0.9)
                axes[0].set_title(f'Raw vs. Cleaned Signal (Channel {channel_to_plot})')
                axes[0].set_ylabel('Amplitude')
                axes[0].legend()

                # Plot summary features (e.g., mean and std amplitude across channels)
                channel_indices = np.arange(n_channels)
                axes[1].bar(channel_indices - 0.2, summary_features["mean_amplitude"], width=0.4, label='Mean Amplitude', color='skyblue')
                axes[1].bar(channel_indices + 0.2, summary_features["std_amplitude"], width=0.4, label='Std Amplitude', color='lightcoral')
                axes[1].set_title('Summary Features Across Channels')
                axes[1].set_ylabel('Value')
                axes[1].set_xlabel('Channel')
                axes[1].legend()
                axes[1].set_xticks(channel_indices[::8]) # Show fewer x-ticks for clarity

                # Display peak-to-peak as text on a subplot or an additional plot
                axes[2].plot(channel_indices, summary_features["peak_to_peak"], marker='o', linestyle='-', color='purple')
                axes[2].set_title('Peak-to-Peak Amplitude Across Channels')
                axes[2].set_xlabel('Channel')
                axes[2].set_ylabel('Amplitude Range')

                plt.tight_layout()
                plt.savefig('brain_signal_analysis_results.png')
                plt.close(fig) # Close the figure to free up memory

                print(f"{stage_name_3} completed successfully. Figure saved as 'brain_signal_analysis_results.png'.")
                print("Data passed from Stage 2 to Stage 3 successfully.")

            except Exception as e:
                error_msg = f"Error in {stage_name_3}: {e}"
                print(f"FAILURE: {error_msg}")
                workflow_log["errors_caught"].append({"stage": stage_name_3, "message": str(e), "traceback": traceback.format_exc()})
        else:
            print(f"Skipping {stage_name_3} due to previous errors or missing data.")
    finally: # This finally belongs to the outer try and handles timing for Stage 3
        workflow_log["stage_processing_times"][stage_name_3] = time.time() - start_time_3

    # --- Stage 4: Documenting Research ---
    stage_name_4 = "Stage 4: Documenting Research"
    start_time_4 = time.time()
    try:
        print(f"\n{'='*50}\n{stage_name_4}: Recording analysis settings, times, and errors.\n{'='*50}")

        log_filename = "research_workflow_log.json"
        with open(log_filename, 'w') as f:
            json.dump(workflow_log, f, indent=4)

        print(f"{stage_name_4} completed successfully. Log saved to '{log_filename}'.")
        # No explicit data passing to Stage 4 as it's the final documentation step

    except Exception as e:
        error_msg = f"Error in {stage_name_4}: {e}"
        print(f"FAILURE: {error_msg}")
        workflow_log["errors_caught"].append({"stage": stage_name_4, "message": str(e), "traceback": traceback.format_exc()})

    finally:
        workflow_log["stage_processing_times"][stage_name_4] = time.time() - start_time_4

    print("\nNeuroscience Workflow Finished.")
    if workflow_log["errors_caught"]:
        print(f"WARNING: {len(workflow_log['errors_caught'])} error(s) occurred during the workflow. Check log for details.")
    else:
        print("All stages completed without errors.")
    print("Total Workflow Time: {:.2f} seconds".format(sum(workflow_log["stage_processing_times"].values())))

# Run the complete workflow
if __name__ == '__main__':
    run_neuroscience_workflow()


Starting Neuroscience Workflow...

Stage 1: Obtaining Data: Generating simulated 64-channel brain recording.
Stage 1: Obtaining Data completed successfully. Generated data shape: (64, 10000)

Stage 2: Analyzing Data: Cleaning data and extracting features.
Stage 2: Analyzing Data completed successfully. Extracted features for 64 channels.
Data passed from Stage 1 to Stage 2 successfully.

Stage 3: Visualizing Results: Creating high-quality figure.
Stage 3: Visualizing Results completed successfully. Figure saved as 'brain_signal_analysis_results.png'.
Data passed from Stage 2 to Stage 3 successfully.

Stage 4: Documenting Research: Recording analysis settings, times, and errors.
Stage 4: Documenting Research completed successfully. Log saved to 'research_workflow_log.json'.

Neuroscience Workflow Finished.
All stages completed without errors.
Total Workflow Time: 0.91 seconds


### ✍️ Part A Reflection
*Double-click this cell to write your response.*

* **What I observed:** Look at the status messages printed between stages. Did each stage receive data in the format it expected? Were any errors caught and logged?
* **Connecting to key concepts:** This integrated workflow demonstrates mastery of all 50 concepts in the Reference Guide. In your own words, describe what happens if Stage 2 (Analyzing Data) tries to pass its results to Stage 3 (Visualizing Results) in the wrong format — and how the error-handling safeguards in the workflow help you find this quickly.

---

## 📑 You Are Ready for the Onsite Workshop

Completing this notebook means you have worked through all six foundational areas:

1. **Notebook 1:** How brain and behavioral data is captured and how computers organize it
2. **Notebook 2:** Working with larger datasets and running analyses efficiently
3. **Notebook 3:** Accessing national research computing resources and connecting securely
4. **Notebook 4:** Cleaning signals and building reliable workflows
5. **Notebook 5:** Finding patterns and making workflows reproducible
6. **Notebook 6:** Connecting all four stages into a complete integrated workflow

The onsite workshop will apply these foundations across three progressively more demanding scenarios — a single-participant workflow, a group of 50 participants, and finally a small portable edge device. Everything you have practiced here will be directly relevant.


---

## 📖 Reference Guide: 50 Key Concepts for Neuroscience Workflows

Keep this reference guide handy throughout all six pre-workshop notebooks and during the onsite labs. These concepts form the foundation of the workshop's hands-on activities.

### 🧠 Neuroscience & Research Applications (Concepts 1–25)
1. **Brain-Machine Interface (BMI):** A system that connects the brain directly to an external device — bypassing damaged nerves or muscles — so that brain signals can control a robotic limb, cursor, or other tool.
2. **Non-Invasive (EEG) vs. Invasive (Intracortical) Sensors:** Scalp electrodes (EEG) are easy to use but pick up blurry, averaged signals through the skull. Implanted microelectrode arrays record from individual neurons with much greater precision, but require surgery.
3. **Signal Delay (Latency):** The time gap between a brain signal being recorded and a device responding to it. Delays longer than about 50–100 milliseconds feel unnatural to the user of a prosthetic device.
4. **Decoder:** A mathematical or statistical model that translates continuous brain signal patterns into a useful output — such as movement direction, cursor position, or speech intent.
5. **fMRI:** Functional Magnetic Resonance Imaging — a brain scanning method that measures blood oxygen levels as a stand-in for neural activity. When neurons become active, they consume more oxygen, causing a detectable change in the local blood signal.
6. **Voxel:** A small 3D cube of brain tissue in an fMRI image — the brain-imaging equivalent of a pixel. Each voxel contains millions of neurons.
7. **Head Movement Correction:** A processing step that aligns brain scan images across time, compensating for small movements the participant made during the recording session.
8. **Open-Loop vs. Closed-Loop Systems:** An open-loop device follows a fixed program regardless of what the user's brain is doing. A closed-loop device continuously reads incoming signals and adjusts its output in real time based on what it detects.
9. **Deep Brain Stimulation (DBS):** A treatment for conditions like Parkinson's disease in which a small implanted device delivers electrical pulses to specific brain regions to reduce tremor and improve movement.
10. **EMG (Electromyogram):** A recording of the electrical signals produced by muscles when they contract. Often used alongside brain recordings to verify whether a behavioral response actually occurred.
11. **Continuous Recordings vs. Event Markers:** A continuous recording captures an uninterrupted time series of measurements. Event markers are specific timestamps that label when something important happened — such as when a stimulus appeared or a button was pressed.
12. **Signal Drift:** The gradual change in a recorded signal over time, not due to the brain, but due to electrode movement, tissue changes, or electronic drift. Workflows need to account for this to keep analysis accurate over long sessions.
13. **Open Data Repositories:** Publicly accessible online archives (such as DANDI, OpenNeuro, or the Human Connectome Project) where researchers share their raw data so others can analyze or replicate their findings.
14. **Automated Data Download (API):** A method of downloading data directly inside a script using a standardized web link, rather than clicking through a website manually. This makes data access reproducible and audit-able.
15. **Artifact Removal:** The process of identifying and removing unwanted signals — such as electrical noise from the building, muscle movements, or eye blinks — from a raw brain recording before analysis.
16. **Downsampling:** Reducing the number of data points per second in a recording, to save memory and processing time, when the extra detail is not needed for the analysis.
17. **Standard Data Formats (BIDS):** A community-agreed system for naming and organizing brain imaging files so that any researcher or software tool can understand the structure without needing special instructions.
18. **Spectrogram:** A visual display showing how the frequency content of a signal changes over time — useful for seeing when the brain shifts between different rhythmic states such as sleep stages or attention levels.
19. **Nyquist Rule:** A fundamental rule of digital recording: to accurately capture a signal, you must record at least twice as fast as the highest frequency in that signal. Recording too slowly creates false patterns called aliasing.
20. **Local Field Potential (LFP):** An electrical recording that reflects the combined activity of a small cluster of nearby neurons — capturing the general activity level of a local brain region rather than individual cells.
21. **Signal Transfer Function:** A mathematical description of how an input signal is transformed into an output signal by a processing step — useful for predicting what a filter or decoder will do to any given input.
22. **Spike Sorting:** The process of separating a mixed electrical recording from multiple nearby neurons into individual neuron signals, based on the distinct shape of each neuron's electrical discharge.
23. **Machine Learning for Neural Decoding:** Using statistical learning algorithms to automatically find patterns in brain signal data that predict behavior, movement intent, or cognitive state.
24. **Sensory Feedback:** Sending information back to the user of a brain-machine interface — for example, delivering a gentle electrical sensation to the skin to simulate the feeling of touching an object with a prosthetic hand.
25. **Neural Plasticity:** The brain's ability to reorganize its connections over time. Relevant to BMI research because users can learn to control devices more accurately with practice as their brain adapts.

### 💻 Computing & Workflow Fundamentals (Concepts 26–50)
26. **Working Memory (RAM) vs. Permanent Storage:** RAM (working memory) holds data only while the computer is on — it is extremely fast but temporary. The hard drive stores files permanently but is much slower to access.
27. **Processor Core:** A single computing unit inside a central processor (CPU). Most modern computers have multiple cores, allowing several tasks to run at the same time.
28. **CPU vs. GPU:** A CPU handles complex, varied tasks one at a time across a few powerful cores. A GPU handles simple, repetitive tasks across thousands of smaller cores simultaneously — useful for image processing and machine learning.
29. **Memory Overflow:** What happens when a script tries to load more data into working memory (RAM) than the computer has available — causing the program to crash.
30. **Motherboard:** The main circuit board inside a computer that connects all the components — processor, memory, storage, and network — so they can communicate with each other.
31. **Processor Slowdown (Thermal Throttling):** When a processor gets too hot, it automatically slows itself down to prevent damage. This can cause unexpected slowdowns during long analysis runs.
32. **High-Speed Processor Cache:** A small, extremely fast memory area built directly into the processor, used to store frequently needed values so they do not have to be fetched from RAM repeatedly.
33. **Local vs. Cloud Computing:** Running your analysis on the computer in front of you (local) versus running it on a remote server accessed over the internet (cloud). Cloud computing allows access to much more memory and processing power.
34. **Temporary Cloud Workspace:** Cloud computing environments like Google Colab provide a temporary workspace that is automatically cleared when you close the session. Any files you need to keep must be saved to permanent storage before the session ends.
35. **Internet Speed as a Bottleneck:** When downloading large research datasets, the speed of your internet connection often limits how fast data can arrive — regardless of how fast your computer itself is.
36. **Organizing Code into Stages:** Separating a workflow into clearly defined, independent stages — such as one stage for loading data, one for analysis, and one for visualization — makes it much easier to find and fix problems.
37. **Whole Numbers vs. Decimal Numbers in Computing:** Computers store whole numbers (integers) very efficiently. Decimal numbers (floating-point) require more memory and processing time. Choosing the right type for your data can affect both speed and accuracy.
38. **Automated Data Access (API):** A standardized connection that allows one piece of software to request data or services from another automatically — for example, a script that downloads data from a research repository without any manual steps.
39. **Settings Files (JSON/YAML):** Lightweight text files used to store configuration settings, metadata, and parameters for a workflow — making it easy to share, reproduce, or adjust an analysis without changing the code itself.
40. **Processing Delay:** The time between when data arrives in your workflow and when your analysis produces a result. In real-time recording systems, keeping this delay short is critical.
41. **Keeping Stages Independent:** Designing a workflow so that the data analysis stage does not depend on the visualization stage, and vice versa. This means you can update or replace one stage without breaking the others.
42. **Text Files vs. Optimized Data Files:** Saving data as plain text (such as CSV) is easy to read in a spreadsheet but very slow for large datasets. Optimized formats (such as HDF5 or NumPy binary files) are much faster to load and take up less disk space.
43. **Simultaneous Processing (Parallel Computing):** Splitting a large task — such as analyzing 50 participants — into smaller chunks that run at the same time across multiple processor cores, rather than one after another.
44. **Hidden Configuration Settings:** Values that a workflow needs — such as access keys for a data repository — that are stored securely in the operating system rather than written directly into the code, to prevent accidental exposure.
45. **Software Libraries (Dependencies):** Pre-built collections of code (such as NumPy, SciPy, or scikit-learn) that provide ready-made tools for common tasks — so you do not have to write mathematical functions from scratch.
46. **Code Version Tracking (Git):** A system that records every change made to a set of code files over time, along with who made the change and when. This allows teams to collaborate and to restore earlier versions if something goes wrong.
47. **Error Handling:** Code that anticipates things going wrong — such as a missing file or a calculation that produces an undefined result — and responds gracefully rather than crashing the entire workflow.
48. **Data Array:** A structured list of numbers organized so that a computer can perform calculations on all of them efficiently — the basic building block of scientific data analysis.
49. **Data Backlog:** What happens when data arrives faster than your workflow can process it — causing a growing queue that eventually uses up available memory.
50. **Protecting Original Data:** A core rule of reproducible research: never modify your raw data files. Always write processed results to a new, separate file so the original record remains intact.


---

## ✅ Grand Portfolio Registry: Confirming Your Complete Preparation

*Check off each handbook below once you have completed its activities and filled in its self-check table.*

| Pre-Workshop Notebook | Concept Areas Covered | Status |
|---|---|---|
| **Notebook 01: Foundations** | Concepts 1–10, 26–35 | [ Completed ] |
| **Notebook 02: Larger Datasets** | Concepts 11–14, 36–43 | [ Completed ] |
| **Notebook 03: National Research Computing Access** | Own 15-concept FABRIC Reference Guide (not part of this list) | [ Completed ] |
| **Notebook 04: Signal Processing** | Concepts 15–21, 45–47 | [ Completed ] |
| **Notebook 05: Analysis & Patterns** | Concepts 22–25, 44, 48–50 | [ Completed ] |
| **Notebook 06: Full Integration** | All 50 Concepts (plus the FABRIC Reference Guide from Notebook 3) | [ Completed ] |
